# Geometric Stroke Transformer

这个 notebook 给出一个起步版的 `encoder-decoder` 笔模型：

- `Text Encoder` 处理 prompt
- `Stroke Decoder` 自回归生成 `{dx, dy, pen_state}`
- token 输入同时包含 `序列位置编码` 和 `绝对几何位置编码`
- self-attention 分数里加入 `相对几何位置 bias`

重点是把架构和前向传播写清楚，先不追求训练技巧或最强性能。

## Design Sketch

一层 decoder block 的结构：

1. 轨迹 token embedding
2. causal self-attention with relative geometric bias
3. cross-attention to text encoder outputs
4. feed-forward network

每个 stroke token 的输入特征：

- `dx, dy`
- `pen_state`
- `x, y`  累计绝对坐标
- `step_length`
- `direction`

In [ ]:
import math
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [ ]:
@dataclass
class ModelConfig:
    vocab_size: int = 256
    d_model: int = 128
    n_heads: int = 8
    ff_mult: int = 4
    num_encoder_layers: int = 2
    num_decoder_layers: int = 3
    max_text_len: int = 64
    max_stroke_len: int = 256
    pen_states: int = 4
    dropout: float = 0.1
    fourier_bands: int = 8


cfg = ModelConfig()
cfg

## Utility Modules

In [ ]:
class FourierFeatures(nn.Module):
    def __init__(self, in_dim: int, num_bands: int):
        super().__init__()
        self.in_dim = in_dim
        self.num_bands = num_bands
        freqs = 2 ** torch.arange(num_bands).float() * math.pi
        self.register_buffer('freqs', freqs, persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.unsqueeze(-1) * self.freqs
        sin = torch.sin(x)
        cos = torch.cos(x)
        return torch.cat([sin, cos], dim=-1).flatten(start_dim=-2)


class AbsoluteGeometryEmbedding(nn.Module):
    def __init__(self, d_model: int, num_bands: int):
        super().__init__()
        self.fourier = FourierFeatures(2, num_bands)
        self.proj = nn.Sequential(
            nn.Linear(2 * num_bands * 2, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
        )

    def forward(self, xy: torch.Tensor) -> torch.Tensor:
        return self.proj(self.fourier(xy))


class StrokeFeatureEmbedding(nn.Module):
    def __init__(self, d_model: int, pen_states: int, num_bands: int):
        super().__init__()
        self.pen_emb = nn.Embedding(pen_states, d_model)
        self.numeric_proj = nn.Sequential(
            nn.Linear(6, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
        )
        self.abs_geo = AbsoluteGeometryEmbedding(d_model, num_bands)

    def forward(self, stroke_feats: torch.Tensor, pen_state: torch.Tensor) -> torch.Tensor:
        # stroke_feats: [B, T, 6] = [dx, dy, x, y, step_length, direction]
        xy = stroke_feats[..., 2:4]
        return self.numeric_proj(stroke_feats) + self.pen_emb(pen_state) + self.abs_geo(xy)

In [ ]:
class RelativeGeometryBias(nn.Module):
    def __init__(self, n_heads: int, num_bands: int):
        super().__init__()
        self.fourier = FourierFeatures(4, num_bands)
        self.mlp = nn.Sequential(
            nn.Linear(4 * num_bands * 2 + 1, 64),
            nn.GELU(),
            nn.Linear(64, n_heads),
        )

    def forward(self, xy: torch.Tensor, direction: torch.Tensor) -> torch.Tensor:
        # xy: [B, T, 2], direction: [B, T, 1]
        rel = xy[:, :, None, :] - xy[:, None, :, :]
        dx = rel[..., 0]
        dy = rel[..., 1]
        r = torch.sqrt(dx * dx + dy * dy + 1e-8)
        theta = torch.atan2(dy, dx)

        geo_feat = torch.stack([dx, dy, r, theta], dim=-1)
        geo_feat = self.fourier(geo_feat)

        dir_rel = direction[:, :, None, :] - direction[:, None, :, :]
        feat = torch.cat([geo_feat, dir_rel], dim=-1)
        bias = self.mlp(feat)
        return bias.permute(0, 3, 1, 2)


## Attention Blocks

这里的关键点：

- self-attention 使用 `causal mask`
- self-attention 额外加 `relative geometric bias`
- cross-attention 对编码器输出做条件对齐

In [ ]:
class GeometricCausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dropout: float, num_bands: int):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
        self.rel_bias = RelativeGeometryBias(n_heads, num_bands)

    def forward(self, x: torch.Tensor, xy: torch.Tensor, direction: torch.Tensor) -> torch.Tensor:
        bsz, seq_len, _ = x.shape
        qkv = self.qkv(x).chunk(3, dim=-1)
        q, k, v = [t.view(bsz, seq_len, self.n_heads, self.head_dim).transpose(1, 2) for t in qkv]

        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        scores = scores + self.rel_bias(xy, direction)

        causal = torch.triu(torch.ones(seq_len, seq_len, device=x.device, dtype=torch.bool), diagonal=1)
        scores = scores.masked_fill(causal[None, None, :, :], float('-inf'))

        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        out = attn @ v
        out = out.transpose(1, 2).contiguous().view(bsz, seq_len, self.d_model)
        return self.out(out)


class CrossAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dropout: float):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.q = nn.Linear(d_model, d_model)
        self.kv = nn.Linear(d_model, 2 * d_model)
        self.out = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, context: torch.Tensor) -> torch.Tensor:
        bsz, seq_len, _ = x.shape
        ctx_len = context.size(1)
        q = self.q(x).view(bsz, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        k, v = self.kv(context).chunk(2, dim=-1)
        k = k.view(bsz, ctx_len, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(bsz, ctx_len, self.n_heads, self.head_dim).transpose(1, 2)

        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        out = attn @ v
        out = out.transpose(1, 2).contiguous().view(bsz, seq_len, self.d_model)
        return self.out(out)


In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model: int, mult: int, dropout: float):
        super().__init__()
        hidden = d_model * mult
        self.net = nn.Sequential(
            nn.Linear(d_model, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, d_model),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class LocalGeometrySummary(nn.Module):
    def __init__(self, d_model: int):
        super().__init__()
        self.conv = nn.Conv1d(d_model, d_model, kernel_size=3, padding=1, groups=1)
        self.proj = nn.Linear(d_model, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        local = self.conv(x.transpose(1, 2)).transpose(1, 2)
        return self.proj(local)


class DecoderBlock(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.local_geom = LocalGeometrySummary(cfg.d_model)
        self.self_attn = GeometricCausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.dropout, cfg.fourier_bands)
        self.cross_attn = CrossAttention(cfg.d_model, cfg.n_heads, cfg.dropout)
        self.ffn = FeedForward(cfg.d_model, cfg.ff_mult, cfg.dropout)
        self.norm1 = nn.LayerNorm(cfg.d_model)
        self.norm2 = nn.LayerNorm(cfg.d_model)
        self.norm3 = nn.LayerNorm(cfg.d_model)
        self.norm4 = nn.LayerNorm(cfg.d_model)

    def forward(self, x: torch.Tensor, xy: torch.Tensor, direction: torch.Tensor, context: torch.Tensor) -> torch.Tensor:
        x = x + self.local_geom(self.norm1(x))
        x = x + self.self_attn(self.norm2(x), xy, direction)
        x = x + self.cross_attn(self.norm3(x), context)
        x = x + self.ffn(self.norm4(x))
        return x


## Encoder-Decoder Model

In [ ]:
class TextEncoder(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.token_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.pos_emb = nn.Embedding(cfg.max_text_len, cfg.d_model)
        layer = nn.TransformerEncoderLayer(
            d_model=cfg.d_model,
            nhead=cfg.n_heads,
            dim_feedforward=cfg.d_model * cfg.ff_mult,
            dropout=cfg.dropout,
            activation='gelu',
            batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=cfg.num_encoder_layers)

    def forward(self, token_ids: torch.Tensor) -> torch.Tensor:
        positions = torch.arange(token_ids.size(1), device=token_ids.device)
        x = self.token_emb(token_ids) + self.pos_emb(positions)[None, :, :]
        return self.encoder(x)


class GeometricStrokeTransformer(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        self.encoder = TextEncoder(cfg)
        self.stroke_emb = StrokeFeatureEmbedding(cfg.d_model, cfg.pen_states, cfg.fourier_bands)
        self.step_pos_emb = nn.Embedding(cfg.max_stroke_len, cfg.d_model)
        self.decoder = nn.ModuleList([DecoderBlock(cfg) for _ in range(cfg.num_decoder_layers)])
        self.final_norm = nn.LayerNorm(cfg.d_model)
        self.delta_head = nn.Linear(cfg.d_model, 2)
        self.pen_head = nn.Linear(cfg.d_model, cfg.pen_states)

    def forward(
        self,
        text_tokens: torch.Tensor,
        stroke_feats: torch.Tensor,
        pen_state: torch.Tensor,
    ):
        context = self.encoder(text_tokens)

        seq_positions = torch.arange(stroke_feats.size(1), device=stroke_feats.device)
        x = self.stroke_emb(stroke_feats, pen_state) + self.step_pos_emb(seq_positions)[None, :, :]
        xy = stroke_feats[..., 2:4]
        direction = stroke_feats[..., 5:6]

        for block in self.decoder:
            x = block(x, xy, direction, context)

        x = self.final_norm(x)
        delta = self.delta_head(x)
        pen_logits = self.pen_head(x)
        return {
            'context': context,
            'hidden': x,
            'pred_delta': delta,
            'pred_pen_logits': pen_logits,
        }

## Dummy Batch

这里构造一个很小的 batch，只为了跑通前向传播。

stroke_feats 的 6 维含义是：

1. `dx`
2. `dy`
3. `x`
4. `y`
5. `step_length`
6. `direction`

In [ ]:
def build_dummy_stroke_batch(batch_size: int = 2, seq_len: int = 12):
    deltas = torch.randn(batch_size, seq_len, 2) * 0.1
    xy = deltas.cumsum(dim=1)
    step_len = deltas.norm(dim=-1, keepdim=True)
    direction = torch.atan2(deltas[..., 1], deltas[..., 0]).unsqueeze(-1)
    feats = torch.cat([deltas, xy, step_len, direction], dim=-1)

    pen = torch.zeros(batch_size, seq_len, dtype=torch.long)
    pen[:, 0] = 1   # move
    pen[:, 1:-2] = 0 # draw
    pen[:, -2] = 2   # end_shape
    pen[:, -1] = 3   # end_all
    return feats, pen


batch_size = 2
text_len = 10
stroke_len = 12

text_tokens = torch.randint(0, cfg.vocab_size, (batch_size, text_len), device=device)
stroke_feats, pen_state = build_dummy_stroke_batch(batch_size, stroke_len)
stroke_feats = stroke_feats.to(device)
pen_state = pen_state.to(device)

text_tokens.shape, stroke_feats.shape, pen_state.shape

In [ ]:
model = GeometricStrokeTransformer(cfg).to(device)
out = model(text_tokens, stroke_feats, pen_state)

for key, value in out.items():
    print(f'{key}: {tuple(value.shape)}')

## Reading the Forward Pass

这个模型的前向传播可以概括成：

1. `prompt -> text encoder -> context tokens`
2. `stroke history -> feature embedding + seq pos + abs geo pos`
3. decoder block 重复执行：
   - local geometry summary
   - geometric causal self-attention
   - cross-attention to prompt context
   - FFN
4. 输出每个时间步的 `delta` 和 `pen_state logits`

这里的关键设计点：

- `AbsoluteGeometryEmbedding` 放在 token 输入层
- `RelativeGeometryBias` 放在 self-attention score 层
- self-attention 使用 `causal mask`，cross-attention 不用因果掩码

In [ ]:
# 只看最后一步输出，模拟“预测下一笔”
last_delta = out['pred_delta'][:, -1]
last_pen = out['pred_pen_logits'][:, -1].softmax(dim=-1)

print('last delta prediction:')
print(last_delta)
print('last pen_state probabilities:')
print(last_pen)

## Next Steps

如果你接下来想把它变成更像研究原型的版本，可以继续加：

- teacher forcing + next-step loss
- `pen_state = {draw, move, end_shape, end_all}` 的正式标签
- `is_corner` 或 `distance_to_closure` 作为辅助头
- 更强的相对几何 bias，例如额外编码距离和方向
- shape-level token 或 graph-conditioned token